<a href="https://colab.research.google.com/github/jstyoon96/WPI-AI-Course/blob/main/WPI_week5/lab1/WPI_week5_lab1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Segmentation Model Development

**Audience:** WPI AI Bootcamp students with basic Python experience.

**Estimated time:** 120-150 minutes.

**Clinical disclaimer:** This lab uses a public pet image dataset as a segmentation teaching dataset. The workflow mirrors biomedical segmentation ideas, but the outputs are not clinical measurements.


## Learning Objectives

By the end of this lab, you should be able to:

- Load image-mask pairs for a binary segmentation task.
- Compare a classical thresholding baseline with a small learned model.
- Train a compact U-Net-style model in Colab.
- Evaluate segmentation with Dice and IoU.
- Explain why numerical metrics and visual overlays are both needed.


## Grading And Word Response Submission

This lab is graded out of **100 pts**.

- Notebook execution and artifacts: **60 pts**
- Word response document: **40 pts**

Use this filename for the Word response document:

`WPI_week5_lab1_responses_LastName_FirstName.docx`

Answer the Word response questions in 2-5 sentences each. Keep longer written responses in the Word document rather than in notebook markdown cells.


## Workflow

This lab follows a segmentation development pipeline:

`Image -> Classical Baseline -> Tiny U-Net -> Probability Map -> Threshold -> Mask -> Evaluation`

The classical baseline is intentionally simple. It gives you a transparent reference point before training a learned model.


## Setup

Run this setup cell first. The notebook installs required packages, clones the public course helper repo in Colab, and imports shared data/style helpers.


In [ ]:
#@title Setup course environment
import subprocess
import sys
from pathlib import Path

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "scikit-image",
    "matplotlib",
])

repo_dir = Path("/content/WPI-AI-Course")
if not repo_dir.parent.exists():
    repo_dir = Path("/tmp/WPI-AI-Course")

if not repo_dir.exists():
    subprocess.check_call([
        "git",
        "clone",
        "--depth",
        "1",
        "https://github.com/jstyoon96/WPI-AI-Course.git",
        str(repo_dir),
    ])

sys.path.insert(0, str(repo_dir))

import random
import numpy as np
import matplotlib.pyplot as plt

from skimage.filters import threshold_otsu
from skimage.morphology import binary_closing, disk, remove_small_objects

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset, random_split

from wpi_ai_bootcamp.data import load_oxford_pet_segmentation_subset
from wpi_ai_bootcamp.notebook import make_wpi_overlay, setup_lab

WPI_COLORS = setup_lab()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Setup complete. Device:", DEVICE)


## Data Loading

This lab downloads a compact subset of the Oxford-IIIT Pet segmentation dataset at runtime through `torchvision.datasets.OxfordIIITPet`.

The dataset provides image-trimap pairs. In this lab, the trimap is converted into a binary foreground mask so the task stays focused on binary segmentation.


In [ ]:
images, masks, metadata = load_oxford_pet_segmentation_subset(
    max_samples=96,
    image_size=128,
    binary_foreground=True,
    download=True,
    random_state=42,
)

source = metadata["source"]
print(source.name)
print(source.url)
print("images:", images.shape, images.dtype, float(images.min()), float(images.max()))
print("masks:", masks.shape, masks.dtype, sorted(np.unique(masks).tolist()))


## Hyperparameters

Only change values in this block when the notebook asks you to run a controlled comparison.


In [ ]:
# STUDENT-EDITABLE HYPERPARAMETERS
SEED = 42
LR = 1e-3
BATCH_SIZE = 8
EPOCHS = 3
BASE_CHANNELS = 12
THRESHOLD = 0.5
MORPH_RADIUS = 3
MIN_OBJECT = 30
SHOW_EXAMPLE_INDEX = 0

# TODO: For Part 5, change exactly one value above and record the result in your Word response.


In [ ]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)


## Part 1 — Dataset Visualization

Inspect one image-mask pair before training anything. Look for whether the mask aligns with the visible foreground.


In [ ]:
def show_image_mask_overlay(image, mask, title="Sample"):
    gray = image.squeeze()
    binary = mask.squeeze() > 0.5
    overlay = make_wpi_overlay(gray, binary)

    fig, axes = plt.subplots(1, 3, figsize=(10, 3))
    axes[0].imshow(gray, cmap="gray")
    axes[0].set_title("Image")
    axes[1].imshow(binary, cmap="gray")
    axes[1].set_title("Mask")
    axes[2].imshow(overlay)
    axes[2].set_title("Overlay")
    for ax in axes:
        ax.axis("off")
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()

show_image_mask_overlay(images[SHOW_EXAMPLE_INDEX], masks[SHOW_EXAMPLE_INDEX], "Oxford Pet example")


### Part 1 Assessment — Dataset Inspection (20 pts)

Required notebook output: one image, mask, and overlay figure.

Word response Q1: What makes this a pixel-wise prediction task rather than an image classification task?

Grading criteria: correct output, clear mask interpretation, and a concise connection to segmentation.


## Part 2 — Classical Baseline Segmentation

Use Otsu thresholding plus morphology as a reproducible baseline.


In [ ]:
class SegmentationArrayDataset(Dataset):
    def __init__(self, images, masks):
        self.images = torch.tensor(images, dtype=torch.float32)
        self.masks = torch.tensor(masks, dtype=torch.float32)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        return self.images[idx], self.masks[idx]


def dice_score(gt, pred, eps=1e-8):
    gt = np.asarray(gt, dtype=np.float32).reshape(-1)
    pred = np.asarray(pred, dtype=np.float32).reshape(-1)
    intersection = (gt * pred).sum()
    return (2 * intersection + eps) / (gt.sum() + pred.sum() + eps)


def iou_score(gt, pred, eps=1e-8):
    gt = np.asarray(gt, dtype=np.float32).reshape(-1)
    pred = np.asarray(pred, dtype=np.float32).reshape(-1)
    intersection = (gt * pred).sum()
    union = ((gt + pred) > 0).sum()
    return (intersection + eps) / (union + eps)


def classical_segmentation(image_tensor):
    image_np = image_tensor.squeeze().cpu().numpy()
    threshold = threshold_otsu(image_np)
    pred = image_np > threshold
    pred = binary_closing(pred, disk(MORPH_RADIUS))
    pred = remove_small_objects(pred, min_size=MIN_OBJECT)
    return pred.astype(np.float32)


dataset = SegmentationArrayDataset(images, masks)
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(
    dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(SEED),
)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False)
print("Train samples:", len(train_dataset), "Validation samples:", len(val_dataset))


In [ ]:
baseline_dice = []
baseline_iou = []
for image, mask in val_dataset:
    pred = classical_segmentation(image)
    gt = mask.squeeze().numpy()
    baseline_dice.append(dice_score(gt, pred))
    baseline_iou.append(iou_score(gt, pred))

print("Validation baseline Dice:", float(np.mean(baseline_dice)))
print("Validation baseline IoU :", float(np.mean(baseline_iou)))


### Part 2 Assessment — Classical Baseline (20 pts)

Required notebook output: validation Dice and IoU for the classical baseline.

Word response Q2: What assumption does Otsu thresholding make about image intensities, and where might that assumption fail?

Grading criteria: metrics are produced, the baseline is described accurately, and one limitation is identified.


## Part 3 — Tiny U-Net Training

Train a compact U-Net-style model. The architecture is intentionally small so it can run in Colab during class.


In [ ]:
class TinyUNet(nn.Module):
    def __init__(self, base_channels=12):
        super().__init__()
        self.enc1 = nn.Sequential(
            nn.Conv2d(1, base_channels, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(base_channels, base_channels, 3, padding=1),
            nn.ReLU(),
        )
        self.pool = nn.MaxPool2d(2)
        self.enc2 = nn.Sequential(
            nn.Conv2d(base_channels, base_channels * 2, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(base_channels * 2, base_channels * 2, 3, padding=1),
            nn.ReLU(),
        )
        self.up = nn.ConvTranspose2d(base_channels * 2, base_channels, 2, stride=2)
        self.dec1 = nn.Sequential(
            nn.Conv2d(base_channels * 2, base_channels, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(base_channels, base_channels, 3, padding=1),
            nn.ReLU(),
        )
        self.out = nn.Conv2d(base_channels, 1, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        up = self.up(e2)
        return self.out(self.dec1(torch.cat([up, e1], dim=1)))

model = TinyUNet(BASE_CHANNELS).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
loss_fn = nn.BCEWithLogitsLoss()
train_losses = []

for epoch in range(EPOCHS):
    model.train()
    running = 0.0
    for batch_images, batch_masks in train_loader:
        batch_images = batch_images.to(DEVICE)
        batch_masks = batch_masks.to(DEVICE)
        logits = model(batch_images)
        loss = loss_fn(logits, batch_masks)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        running += loss.item()
    avg_loss = running / len(train_loader)
    train_losses.append(avg_loss)
    print(f"Epoch {epoch + 1}/{EPOCHS} - loss: {avg_loss:.4f}")

checkpoint_path = "best_unet.pt"
torch.save({"model_state_dict": model.state_dict(), "base_channels": BASE_CHANNELS}, checkpoint_path)
print("Saved checkpoint:", checkpoint_path)


In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(range(1, EPOCHS + 1), train_losses, marker="o", color=WPI_COLORS["crimson"])
plt.xlabel("Epoch")
plt.ylabel("Training loss")
plt.title("Tiny U-Net Training Loss")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


### Part 3 Assessment — U-Net Training (20 pts)

Required notebook output: training loss by epoch and saved `best_unet.pt` checkpoint.

Word response Q3: If training loss decreases but the overlay still looks poor, what might that suggest?

Grading criteria: model trains without errors, checkpoint is created, and the response distinguishes optimization from segmentation quality.


## Part 4 — Evaluation

Convert model probabilities to binary masks and compare U-Net performance with the baseline.


In [ ]:
def predict_mask(model, image_tensor, threshold=0.5):
    model.eval()
    with torch.no_grad():
        x = image_tensor.unsqueeze(0).to(DEVICE)
        prob = torch.sigmoid(model(x)).cpu().squeeze().numpy()
    return (prob > threshold).astype(np.float32), prob

unet_dice = []
unet_iou = []
saved_example = None
for i, (image, mask) in enumerate(val_dataset):
    pred, prob = predict_mask(model, image, THRESHOLD)
    gt = mask.squeeze().numpy()
    unet_dice.append(dice_score(gt, pred))
    unet_iou.append(iou_score(gt, pred))
    if i == min(SHOW_EXAMPLE_INDEX, len(val_dataset) - 1):
        saved_example = (image, mask, classical_segmentation(image), pred, prob)

print("Validation U-Net Dice:", float(np.mean(unet_dice)))
print("Validation U-Net IoU :", float(np.mean(unet_iou)))


In [ ]:
plt.figure(figsize=(6, 4))
plt.bar(["Baseline", "Tiny U-Net"], [np.mean(baseline_dice), np.mean(unet_dice)], color=[WPI_COLORS["gray"], WPI_COLORS["crimson"]])
plt.ylabel("Dice score")
plt.ylim(0, 1)
plt.title("Validation Dice Comparison")
plt.tight_layout()
plt.show()

image, mask, baseline_pred, unet_pred, prob = saved_example
fig, axes = plt.subplots(1, 4, figsize=(12, 3))
axes[0].imshow(image.squeeze(), cmap="gray")
axes[0].set_title("Image")
axes[1].imshow(mask.squeeze(), cmap="gray")
axes[1].set_title("Ground truth")
axes[2].imshow(baseline_pred, cmap="gray")
axes[2].set_title("Baseline")
axes[3].imshow(unet_pred, cmap="gray")
axes[3].set_title("Tiny U-Net")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()


### Part 4 Assessment — Evaluation (20 pts)

Required notebook output: U-Net Dice/IoU, comparison bar chart, and qualitative prediction figure.

Word response Q4: Why is visual inspection still important even when Dice and IoU are available?

Grading criteria: metrics and plots are present, and the response identifies a concrete metric limitation.


## Part 5 — One Controlled Comparison

Change exactly one parameter and compare the metric. Keep all other settings identical.

Recommended first comparison: change only `THRESHOLD` below.


In [ ]:
# TODO: Change only this value for the required controlled comparison.
COMPARISON_THRESHOLD = 0.60

comparison_dice = []
comparison_iou = []
for image, mask in val_dataset:
    pred, _ = predict_mask(model, image, COMPARISON_THRESHOLD)
    gt = mask.squeeze().numpy()
    comparison_dice.append(dice_score(gt, pred))
    comparison_iou.append(iou_score(gt, pred))

print("Original THRESHOLD:", THRESHOLD)
print("Comparison THRESHOLD:", COMPARISON_THRESHOLD)
print("Original U-Net Dice:", float(np.mean(unet_dice)))
print("Comparison Dice:", float(np.mean(comparison_dice)))
print("Original U-Net IoU:", float(np.mean(unet_iou)))
print("Comparison IoU:", float(np.mean(comparison_iou)))


### Part 5 Assessment — Controlled Comparison (20 pts)

Required notebook output: original and comparison values for Dice and/or IoU.

Word response Q5: Which single parameter did you change, and how did the change affect segmentation quality?

Grading criteria: exactly one parameter changes, metrics are compared fairly, and the interpretation is tied to the observed result.


## Optional Challenge

Try changing `BASE_CHANNELS` and retraining from the beginning. Compare runtime, loss, Dice, and visual quality. Do not mix this result with the required one-variable comparison unless your instructor asks you to.


## Attribution

- Data: Oxford-IIIT Pet dataset, loaded with `torchvision.datasets.OxfordIIITPet` using segmentation trimaps.
- Dataset page: https://www.robots.ox.ac.uk/~vgg/data/pets/
- PyTorch loader documentation: https://docs.pytorch.org/vision/main/generated/torchvision.datasets.OxfordIIITPet.html
- Citation: O. M. Parkhi, A. Vedaldi, A. Zisserman, and C. V. Jawahar, *Cats and Dogs*, IEEE Conference on Computer Vision and Pattern Recognition, 2012.
- License note: The Oxford dataset page lists Creative Commons Attribution-ShareAlike 4.0 International for commercial/research download; image copyrights remain with original owners.
- Libraries: NumPy, Matplotlib, scikit-image, PyTorch, torchvision, and course helper code.
